# AEGIS-SQL × Qwen2.5-Coder 1.5B

LitE-SQL-inspired pretrained baseline. **Runtime → Change runtime type → GPU**를 먼저 선택하세요.

이 노트북은 base Qwen과 고정된 `aegis-qwen-flywheel-v1` snapshot으로 QLoRA한 모델을 **같은 KorFin evaluator**로 비교합니다. 성능 수치는 실제 실행 결과가 생긴 뒤에만 사용합니다. 과거 5.3M AegisLM은 원본 학습 JSONL이 보존되지 않아 이번 paired comparison에 포함하지 않습니다.

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime을 선택한 뒤 다시 실행하세요.')
print('GPU:', torch.cuda.get_device_name(0))

## 1. 저장소 준비
이미 clone한 세션이라도 재현성을 위해 main을 새로 받습니다.

In [ ]:
!rm -rf /content/aegis-sql
!git clone -q https://github.com/sokldjs554/aegis-sql.git /content/aegis-sql
%cd /content/aegis-sql
!git rev-parse HEAD

## 2. 먼저 smoke run
64 train / 32 dev / 10 eval item으로 wiring과 GPU stack을 확인합니다. 이 결과는 포트폴리오 성능으로 쓰지 않습니다.

In [ ]:
!SMOKE=1 bash scripts/run_qwen_colab.sh

## 3. Full run
Smoke가 성공한 뒤 실행합니다. Git에 고정한 train 9,000 / dev 1,153 snapshot을 사용하고, base → QLoRA → adapted 순서로 전체 answerable KorFin 90문항을 평가합니다.

In [ ]:
!bash scripts/run_qwen_colab.sh

## 4. 결과 확인
검증 스크립트가 90문항 전체, 학습 시간, base/adapted EX, difficulty별 EX, latency, peak CUDA memory가 모두 있을 때만 full run을 완료 처리합니다.

In [ ]:
import json
from pathlib import Path
p = Path('reports/qwen2.5-coder-1.5b-full-summary.json')
d = json.loads(p.read_text())
print('portfolio evidence ready =', d['portfolio_evidence_ready'])
print('GPU =', d['runtime']['gpu']['name'])
print('training seconds =', d['training']['wall_clock_seconds'])
print('training peak CUDA GiB =', d['training']['peak_cuda_memory_gib'])
for label in ('base', 'adapted'):
    row = d[label]
    print('\n', label.upper())
    print('EX =', f"{row['execution_accuracy']:.1%}")
    print('difficulty =', {k: f"{v['ex']:.1%}" for k, v in row['by_difficulty'].items()})
    print('latency_ms =', row['latency_ms'])
    print('peak CUDA GiB =', row['peak_cuda_memory_gib'])
print('\ndelta EX percentage points =', d['delta_ex_percentage_points'])

## 5. 원본 결과 보존
런타임이 종료되기 전에 summary, base/adapted row-level 결과, 학습 manifest가 든 ZIP을 내려받습니다. 어댑터 가중치는 포함하지 않습니다.

In [ ]:
from google.colab import files
files.download('reports/qwen2.5-coder-1.5b-full-results.zip')

## 다음 단계
1.5B full 결과를 먼저 고정한 뒤에만 3B를 같은 데이터·LoRA 설정으로 반복합니다. GPU 자원 때문에 batch/max-length 등을 바꾸면 동일 실험으로 섞지 말고 별도 run으로 기록합니다.